In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Installare e usare i plugin del transpiler

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato con i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Per facilitare lo sviluppo e il riutilizzo di codice di transpilazione personalizzato da parte della comunità Qiskit, l'SDK Qiskit supporta un'interfaccia a plugin che consente a pacchetti Python di terze parti di dichiarare che forniscono funzionalità di transpilazione estesa accessibili tramite Qiskit.

Al momento, i plugin di terze parti possono fornire funzionalità di transpilazione estesa in tre modi:

- Un [plugin per la fase del transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins) fornisce un pass manager che può essere usato al posto di una delle [6 fasi](transpiler-stages) di un pass manager staged predefinito: `init`, `layout`, `routing`, `translation`, `optimization` e `scheduling`.
- Un [plugin di sintesi unitaria](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) fornisce funzionalità estesa per la sintesi di gate unitari.
- Un [plugin di sintesi ad alto livello](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) fornisce funzionalità estesa per la sintesi di "oggetti ad alto livello" come funzioni lineari o operatori di Clifford. Gli oggetti ad alto livello sono rappresentati da sottoclassi della classe [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation).

Il resto della pagina descrive come elencare i plugin disponibili, installarne di nuovi e utilizzarli.
## Elencare i plugin disponibili e installarne di nuovi
Qiskit include già alcuni plugin integrati per la transpilazione. Per installarne altri, puoi usare il tuo gestore di pacchetti Python. Ad esempio, potresti eseguire `pip install qiskit-toqm` per installare il plugin di routing [Qiskit TOQM](https://github.com/qiskit-toqm/qiskit-toqm). Diversi plugin di terze parti fanno parte dell'[ecosistema Qiskit](https://qiskit.github.io/ecosystem/#transpiler_plugin).

### Elencare i plugin disponibili per le fasi del transpiler
Usa la funzione [list_stage_plugins](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#qiskit.transpiler.preset_passmanagers.plugin.list_stage_plugins), passando il nome della fase di cui vuoi elencare i plugin.

In [1]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

In [2]:
list_stage_plugins("routing")

['basic', 'default', 'lookahead', 'none', 'sabre']

 If `qiskit-toqm` were installed, then `toqm` would appear in the list of `routing` plugins.

### List available unitary synthesis plugins

Use the [unitary_synthesis_plugin_names](/docs/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.unitary_synthesis_plugin_names) function.

In [3]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

### List available high-level synthesis plugins

Use the [high_level_synthesis_plugin_names](/docs/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.high_level_synthesis_plugin_names) function, passing the name of the type of "high-level object" to be synthesized. The name corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the [Operation](/docs/api/qiskit/qiskit.circuit.Operation) class representing the type of object being synthesized.

In [4]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

Se `qiskit-toqm` fosse installato, `toqm` apparirebbe nell'elenco dei plugin di `routing`.
### Elencare i plugin di sintesi unitaria disponibili
Usa la funzione [unitary_synthesis_plugin_names](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.unitary_synthesis_plugin_names).

In [5]:
from qiskit.transpiler.passes.synthesis.plugin import (
    HighLevelSynthesisPluginManager,
)

HighLevelSynthesisPluginManager().plugins.names()

['FullAdder.default',
 'FullAdder.ripple_c04',
 'FullAdder.ripple_v95',
 'HalfAdder.default',
 'HalfAdder.qft_d00',
 'HalfAdder.ripple_c04',
 'HalfAdder.ripple_r25',
 'HalfAdder.ripple_v95',
 'IntComp.default',
 'IntComp.noaux',
 'IntComp.twos',
 'ModularAdder.default',
 'ModularAdder.modular_v17',
 'ModularAdder.qft_d00',
 'ModularAdder.ripple_c04',
 'ModularAdder.ripple_v95',
 'Multiplier.cumulative_h18',
 'Multiplier.default',
 'Multiplier.qft_r17',
 'PauliEvolution.default',
 'PauliEvolution.rustiq',
 'WeightedSum.default',
 'annotated.default',
 'clifford.ag',
 'clifford.bm',
 'clifford.default',
 'clifford.greedy',
 'clifford.layers',
 'clifford.lnn',
 'linear_function.default',
 'linear_function.kms',
 'linear_function.pmh',
 'mcmt.default',
 'mcmt.noaux',
 'mcmt.vchain',
 'mcmt.xgate',
 'mcx.1_clean_b95',
 'mcx.1_clean_kg24',
 'mcx.1_dirty_kg24',
 'mcx.2_clean_kg24',
 'mcx.2_dirty_kg24',
 'mcx.default',
 'mcx.gray_code',
 'mcx.n_clean_m15',
 'mcx.n_dirty_i15',
 'mcx.noaux_hp24'

## Use a plugin

In this section, we show how to use transpiler plugins. In the code examples, we use plugins that come with Qiskit, but plugins installed from third-party packages are used the same way.

### Use a transpiler stage plugin

To use a transpiler stage plugin, specify its name with the appropriate argument to [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). The argument is formed by appending `_method` to the name of the transpilation stage. For example, to use the `lookahead` routing plugin, we would specify `lookahead` for the `routing_method` argument:

In [6]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, routing_method="lookahead"
)

### Elencare i plugin di sintesi ad alto livello disponibili
Usa la funzione [high_level_synthesis_plugin_names](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.high_level_synthesis_plugin_names), passando il nome del tipo di "oggetto ad alto livello" da sintetizzare. Il nome corrisponde all'attributo [`name`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation#name) della classe [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) che rappresenta il tipo di oggetto da sintetizzare.

In [7]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    unitary_synthesis_method="sk",
    unitary_synthesis_plugin_config=dict(
        basis_gates=["cz", "id", "rz", "sx", "x"]
    ),
)

Unitary synthesis is used in the `init`, `translation`, and `optimization` stages of the staged pass manager returned by [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or used in [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). See [Transpiler stages](transpiler-stages) for a description of these stages.

Use the `unitary_synthesis_plugin_config` argument, a free-form dictionary, to pass options for the unitary synthesis method. The documentation of the synthesis method should explain the options it supports. See [this list](/docs/api/qiskit/transpiler_synthesis_plugins#unitary-synthesis-plugins-2) for links to the documentation of the built-in unitary synthesis plugins.

### Use a high-level synthesis plugin

First, create an [HLSConfig](/docs/api/qiskit/qiskit.transpiler.passes.HLSConfig) to
store the names of the plugins to use for various high-level objects. For example:

In [8]:
from qiskit.transpiler.passes import HLSConfig

hls_config = HLSConfig(clifford=["layers"], linear_function=["pmh"])

Puoi usare la classe [HighLevelSynthesisPluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPluginManager) per elencare i nomi di tutti i plugin di sintesi ad alto livello:

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, hls_config=hls_config
)

High-level synthesis is used in the `init`, `translation`, and `optimization` stages of the staged pass manager returned by [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or used in [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). See [Transpiler stages](transpiler-stages) for a description of these stages.

## Next steps

<Admonition type="tip" title="Recommendation">
    - [Create a transpiler plugin](./create-transpiler-plugin).
    - Check out the [tutorials](/docs/tutorials) for examples of transpiling and running quantum circuits.
</Admonition>